# CSIRO Image2Biomass: ConvNeXt-V2 Large 5-Fold Dual-Stream Training

This notebook trains the **3rd-Place Core Innovation: ConvNeXt-V2 Large** (`convnextv2_large`, 198M parameters, 1536 feature dimension) integrated into our **Dual-Stream Cross-View Attention Architecture**.

### 🌟 Why ConvNeXt-V2 Large + DINO ViT is the Top-Tier Breakthrough
- **Architectural Diversity**: ViT operates on discrete patch tokens with global self-attention. ConvNeXt-V2 is a pure modern convolutional network featuring 7x7 depthwise convolutions and **Global Response Normalization (GRN)**. Combining their predictions cancels out ViT patch artifacts and CNN boundary biases.
- **3rd-Place Evidence**: The 3rd-place competition team demonstrated that ConvNeXt-V2 Large was their single strongest individual model, and ensembling ConvNeXt-V2 with ViT delivered their largest single leaderboard jump.
- **Dual-Stream 1:1 Square Partitioning**: High-resolution $2000 \times 1000$ images are split into natural Left and Right $1000 \times 1000$ views and processed at $512 \times 512$.
- **Cross-View Multi-Head Attention (1536-dim)**: 8-head self-attention fuses Left and Right view features across the pasture plot seam.
- **Auxiliary Interval Classification (7-bin UEPNet)**: Joint continuous regression + discrete interval cross-entropy for gradient stability.
- **Anti-Leakage Split (Seed 223)**: Grouped by `Sampling_Date` and stratified by `State` (NSW, WA, Tas, Vic) to eliminate temporal/lighting leakage.
- **3rd-Place Kitchen Sink Augmentations**: Vertical 4-strip permutation ($p=0.5$), Random Grayscale ($p=0.2$), Left/Right View Swap ($p=0.5$), and Camera Scaling ($p=0.2$).


In [ ]:
# 1. Environment & Installations
!pip install -q timm

import os
import sys
import glob
import time
import random
import math
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import StratifiedGroupKFold
from torchvision import transforms
import timm

def set_seed(seed=223):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(223)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB')


In [ ]:
# 2. Configuration & Automatic Path Discovery
def find_data_dir():
    candidates = [
        '/kaggle/input/competitions/csiro-biomass',
        '/kaggle/input/csiro-biomass',
        '../input/competitions/csiro-biomass',
        '../input/csiro-biomass',
        './data',
        '.'
    ]
    for c in candidates:
        if os.path.exists(os.path.join(c, 'train.csv')):
            return c
    if os.path.exists('/kaggle/input'):
        for root, _, files in os.walk('/kaggle/input'):
            if 'train.csv' in files:
                return root
    return '.'

class CFG:
    DATA_DIR = find_data_dir()
    TRAIN_CSV = os.path.join(DATA_DIR, 'train.csv')
    TEST_CSV = os.path.join(DATA_DIR, 'test.csv')
    TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train') if os.path.exists(os.path.join(DATA_DIR, 'train')) else DATA_DIR
    TEST_IMG_DIR = os.path.join(DATA_DIR, 'test') if os.path.exists(os.path.join(DATA_DIR, 'test')) else DATA_DIR
    
    # Model parameters: ConvNeXt-V2 Large
    BACKBONE = 'convnextv2_large'   # 198M params, 1536 feature dimension
    MODEL_TAG = 'convnextv2'        # Tag used in saved checkpoint filenames
    IMG_SIZE = 512                  # 512x512 per view (1024x1024 effective field)
    FUSION_DIM = 384
    DROPOUT = 0.3
    
    # Training parameters
    # Batch size 4 + Grad Accum 2 = Effective batch size 8 (rock-solid on 16GB VRAM)
    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 2
    STAGE1_EPOCHS = 8               # Frozen backbone (heads + fusion only)
    STAGE2_EPOCHS = 20              # Full end-to-end fine-tuning
    LR = 3e-4
    BACKBONE_LR_FACTOR = 0.1        # Differential learning rate for backbone (3e-5)
    WEIGHT_DECAY = 0.05
    MAX_GRAD_NORM = 1.0
    N_FOLDS = 5
    
    # Loss & Auxiliary Interval Classification
    NUM_INTERVALS = 7
    CLS_WEIGHT = 0.3
    USE_TTA = True

    # Kitchen Sink Augmentations (3rd-Place Solution Enhancements)
    CAMERA_SCALE_PROB = 0.2
    STRIP_SHUFFLE_PROB = 0.5        # Vertical 4-strip permutation
    VIEW_SWAP_PROB = 0.5            # Left/Right view swap
    GRAYSCALE_PROB = 0.2            # Random grayscale transform

    # Split Strategy (Host-Confirmed Anti-Leakage & Monte Carlo Optimized)
    SEED = 223
    SPLIT_GROUP_COL = 'Sampling_Date'   # Prevents temporal leakage across collection dates
    SPLIT_STRAT_COL = 'State'           # Guarantees balanced state representations (NSW, WA, Tas, Vic)

    # Targets & Official Metric Weights
    TARGET_ORDER = ['Dry_Green_g', 'Dry_Dead_g', 'Dry_Clover_g', 'GDM_g', 'Dry_Total_g']
    OFFICIAL_WEIGHTS = [0.1, 0.1, 0.1, 0.2, 0.5]
    IMAGENET_MEAN = [0.485, 0.456, 0.406]
    IMAGENET_STD = [0.229, 0.224, 0.225]

print(f'Discovered DATA_DIR: {CFG.DATA_DIR}')
print(f'TRAIN_CSV: {CFG.TRAIN_CSV} (Exists: {os.path.exists(CFG.TRAIN_CSV)})')
print(f'TEST_CSV:  {CFG.TEST_CSV} (Exists: {os.path.exists(CFG.TEST_CSV)})')
print(f'Backbone: {CFG.BACKBONE} | Image Size: {CFG.IMG_SIZE}x{CFG.IMG_SIZE} | Effective Batch: {CFG.BATCH_SIZE * CFG.GRAD_ACCUM_STEPS}')
print(f'CV Split Strategy -> Group: {CFG.SPLIT_GROUP_COL} | Stratify: {CFG.SPLIT_STRAT_COL} (Seed: {CFG.SEED})')


In [ ]:
# 3. Metrics, Interval Utilities & Soft Physical Calibration
INTERVAL_BINS = np.array([0.0, 5.0, 15.0, 30.0, 60.0, 100.0, 200.0])

def to_interval_labels(continuous_targets):
    t = np.asarray(continuous_targets)
    labels = np.zeros(t.shape, dtype=np.int64)
    for col in range(t.shape[1]):
        labels[:, col] = np.digitize(t[:, col], INTERVAL_BINS) - 1
    return np.clip(labels, 0, len(INTERVAL_BINS) - 1)

def calculate_competition_r2(y_true, y_pred, weights=CFG.OFFICIAL_WEIGHTS):
    r2_scores = []
    for i in range(y_true.shape[1]):
        yt = y_true[:, i]
        yp = y_pred[:, i]
        ss_res = np.sum((yt - yp) ** 2)
        ss_tot = np.sum((yt - np.mean(yt)) ** 2)
        r2 = 1.0 - (ss_res / ss_tot) if ss_tot > 1e-8 else 0.0
        r2_scores.append(r2)
    return np.sum(np.array(r2_scores) * np.array(weights))

def soft_physics_postprocess(preds_np):
    preds = np.maximum(preds_np.copy(), 0.0)
    green = preds[:, 0]
    dead = preds[:, 1]
    clover = preds[:, 2] * 0.8
    gdm = preds[:, 3]
    total = preds[:, 4]
    
    # 3rd-Place Dead Thatch Fringe Expansion
    dead = np.where(dead > 20.0, dead * 1.1, np.where(dead < 10.0, dead * 0.9, dead))
    
    # Soft mass conservation blends
    gdm_blended = 0.5 * gdm + 0.5 * (green + clover)
    total_blended = 0.5 * total + 0.5 * (green + clover + dead)
    
    return np.maximum(np.column_stack([green, dead, clover, gdm_blended, total_blended]), 0.0)

class WeightedBiomassLoss(nn.Module):
    def __init__(self, loss_weights=CFG.OFFICIAL_WEIGHTS, cls_weight=CFG.CLS_WEIGHT):
        super().__init__()
        self.loss_weights = loss_weights
        self.cls_weight = cls_weight
        self.smooth_l1 = nn.SmoothL1Loss(reduction='none')
        self.ce_loss = nn.CrossEntropyLoss(reduction='mean')
        
    def forward(self, reg_preds, cls_preds, targets_reg, targets_cls):
        total_reg_loss = 0.0
        for i, weight in enumerate(self.loss_weights):
            pred_i = reg_preds[i].squeeze(-1)
            target_i = targets_reg[:, i]
            loss_i = self.smooth_l1(pred_i, target_i).mean()
            total_reg_loss += weight * loss_i
            
        total_cls_loss = 0.0
        for i in range(len(cls_preds)):
            pred_cls_i = cls_preds[i]
            target_cls_i = targets_cls[:, i]
            total_cls_loss += self.ce_loss(pred_cls_i, target_cls_i)
        total_cls_loss = total_cls_loss / len(cls_preds)
        
        total_loss = total_reg_loss + self.cls_weight * total_cls_loss
        return total_loss, total_reg_loss, total_cls_loss


In [ ]:
# 4. Kitchen Sink Dataset & Mass-Conserving Augmentations
def permute_vertical_strips(img_np, n_strips=4, prob=0.5):
    if random.random() > prob:
        return img_np
    h, w, c = img_np.shape
    strip_w = w // n_strips
    strips = [img_np[:, i*strip_w:(i+1)*strip_w, :] for i in range(n_strips)]
    random.shuffle(strips)
    return np.concatenate(strips, axis=1)

class DualStreamBiomassDataset(Dataset):
    def __init__(self, df, img_dir, img_size=512, is_training=True,
                 camera_scaling_prob=0.2, strip_shuffle_prob=0.5, view_swap_prob=0.5, grayscale_prob=0.2):
        self.df = df.reset_index(drop=True)
        self.img_dir = img_dir
        self.img_size = img_size
        self.is_training = is_training
        self.camera_scaling_prob = camera_scaling_prob
        self.strip_shuffle_prob = strip_shuffle_prob
        self.view_swap_prob = view_swap_prob
        self.grayscale_prob = grayscale_prob
        
        self.has_targets = all(c in df.columns for c in CFG.TARGET_ORDER)
        if self.has_targets:
            self.targets_reg = self.df[CFG.TARGET_ORDER].values.astype(np.float32)
            self.targets_cls = to_interval_labels(self.targets_reg)
            
        t_list = [transforms.Resize((img_size, img_size))]
        if is_training:
            if grayscale_prob > 0:
                t_list.append(transforms.RandomGrayscale(p=grayscale_prob))
            t_list.extend([
                transforms.RandomHorizontalFlip(p=0.5),
                transforms.RandomVerticalFlip(p=0.5),
                transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
            ])
        t_list.extend([
            transforms.ToTensor(),
            transforms.Normalize(mean=CFG.IMAGENET_MEAN, std=CFG.IMAGENET_STD)
        ])
        self.transform = transforms.Compose(t_list)

    def __len__(self):
        return len(self.df)

    def _resolve_image_path(self, raw_path):
        if os.path.exists(raw_path): return raw_path
        fname = os.path.basename(raw_path)
        candidates = [
            os.path.join(self.img_dir, raw_path),
            os.path.join(self.img_dir, fname),
            os.path.join(CFG.DATA_DIR, 'train', fname),
            os.path.join(CFG.DATA_DIR, 'test', fname),
            os.path.join(CFG.DATA_DIR, fname),
        ]
        for c in candidates:
            if os.path.exists(c): return c
        return raw_path

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = self._resolve_image_path(row['image_path'])
        img = cv2.imread(path)
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
        h, w, _ = img.shape
        mid = w // 2
        left_np = img[:, :mid, :]
        right_np = img[:, mid:, :]
        
        if self.is_training:
            if self.view_swap_prob > 0 and random.random() < self.view_swap_prob:
                left_np, right_np = right_np, left_np
                
            if self.camera_scaling_prob > 0 and random.random() < self.camera_scaling_prob:
                scale = random.uniform(0.9, 1.1)
                nh, nw = int(h * scale), int((w // 2) * scale)
                left_np = cv2.resize(left_np, (nw, nh))
                right_np = cv2.resize(right_np, (nw, nh))
                
            if self.strip_shuffle_prob > 0:
                left_np = permute_vertical_strips(left_np, n_strips=4, prob=self.strip_shuffle_prob)
                right_np = permute_vertical_strips(right_np, n_strips=4, prob=self.strip_shuffle_prob)
            
        tensor_l = self.transform(Image.fromarray(left_np))
        tensor_r = self.transform(Image.fromarray(right_np))
        
        item = {
            'image_left': tensor_l,
            'image_right': tensor_r,
            'sample_id': row.get('sample_id', row.get('clean_id', f'sample_{idx}')),
        }
        if self.has_targets:
            item['targets'] = torch.tensor(self.targets_reg[idx], dtype=torch.float32)
            item['targets_cls'] = torch.tensor(self.targets_cls[idx], dtype=torch.long)
        return item


In [ ]:
# 5. ConvNeXt-V2 Large DualStreamBiomassModel Architecture
class DualStreamBiomassModel(nn.Module):
    def __init__(self, backbone_name=CFG.BACKBONE, num_targets=5, num_intervals=7, fusion_dim=384, dropout=0.3, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0)
        self.backbone_dim = self.backbone.num_features  # 1536 for convnextv2_large
        
        # 8 Attention heads (1536 / 8 = 192 per head)
        num_heads = 8 if self.backbone_dim % 8 == 0 else 4
        self.cross_view_attn = nn.MultiheadAttention(embed_dim=self.backbone_dim, num_heads=num_heads, dropout=0.1, batch_first=True)
        self.attn_norm = nn.LayerNorm(self.backbone_dim)
        
        self.fusion_mlp = nn.Sequential(
            nn.Linear(self.backbone_dim * 2, fusion_dim),
            nn.LayerNorm(fusion_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        
        self.reg_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, fusion_dim // 2),
                nn.LayerNorm(fusion_dim // 2),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(fusion_dim // 2, 64),
                nn.GELU(),
                nn.Linear(64, 1)
            ) for _ in range(num_targets)
        ])
        
        self.cls_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(fusion_dim, 128),
                nn.LayerNorm(128),
                nn.GELU(),
                nn.Dropout(dropout * 0.5),
                nn.Linear(128, num_intervals)
            ) for _ in range(num_targets)
        ])

    def extract_features(self, x):
        feats = self.backbone(x)
        # Robust spatial pooling: works for ViT [B, N, C] or CNN [B, C, H, W]
        return feats.mean(dim=1) if len(feats.shape) == 3 else feats.mean(dim=[2, 3]) if len(feats.shape) == 4 else feats

    def forward(self, img_left, img_right):
        feat_l = self.extract_features(img_left)   # [B, 1536]
        feat_r = self.extract_features(img_right)  # [B, 1536]
        
        tokens = torch.stack([feat_l, feat_r], dim=1)  # [B, 2, 1536]
        attn_out, _ = self.cross_view_attn(tokens, tokens, tokens)
        tokens = self.attn_norm(tokens + attn_out)
        
        fused = self.fusion_mlp(torch.cat([tokens[:, 0], tokens[:, 1]], dim=-1))  # [B, 384]
        reg_preds = [F.softplus(head(fused)) for head in self.reg_heads]
        cls_preds = [head(fused) for head in self.cls_heads]
        return reg_preds, cls_preds

print(f'ConvNeXt-V2 Model initialized successfully. Feature dim: {1536}')


In [ ]:
# 6. Data Loading, Anti-Leakage Split & Cross-Validation Setup
def load_and_pivot_data(train_csv_path):
    if not os.path.exists(train_csv_path):
        discovered = os.path.join(CFG.DATA_DIR, 'train.csv')
        if os.path.exists(discovered):
            train_csv_path = discovered
        elif os.path.exists('/kaggle/input'):
            for root, _, files in os.walk('/kaggle/input'):
                if 'train.csv' in files:
                    train_csv_path = os.path.join(root, 'train.csv')
                    break
                    
    print(f'Loading data from: {train_csv_path}')
    df = pd.read_csv(train_csv_path)
    if 'target_name' in df.columns:
        df['clean_id'] = df['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
        pivoted = df.pivot(index='clean_id', columns='target_name', values='target').reset_index()
        meta = df[['clean_id', 'image_path', 'State', 'Sampling_Date', 'Species']].drop_duplicates(subset=['clean_id']).reset_index(drop=True)
        df = pd.merge(pivoted, meta, on='clean_id')
        df['sample_id'] = df['clean_id']
        
    if 'Sampling_Date' not in df.columns: df['Sampling_Date'] = 'unknown'
    if 'State' not in df.columns: df['State'] = 'NSW'
    df['Sampling_Date'] = df['Sampling_Date'].astype(str)
    df['State'] = df['State'].astype(str)
    df['State_Sampling_Date'] = df['State'] + '_' + df['Sampling_Date']
    return df

def evaluate(model, val_loader, criterion):
    model.eval()
    val_loss_sum, val_reg_sum, val_cls_sum = 0.0, 0.0, 0.0
    all_val_preds, all_val_targets = [], []
    all_cls_preds, all_cls_targets = [], []
    
    with torch.no_grad():
        for batch in val_loader:
            img_l = batch['image_left'].to(DEVICE)
            img_r = batch['image_right'].to(DEVICE)
            t_reg = batch['targets'].to(DEVICE)
            t_cls = batch['targets_cls'].to(DEVICE)
            
            with torch.amp.autocast('cuda'):
                r, c = model(img_l, img_r)
                loss, l_reg, l_cls = criterion(r, c, t_reg, t_cls)
                
            bs = len(img_l)
            val_loss_sum += loss.item() * bs
            val_reg_sum += l_reg.item() * bs
            val_cls_sum += l_cls.item() * bs
            
            pred_mat = torch.cat(r, dim=1).cpu().numpy()
            all_val_preds.append(pred_mat)
            all_val_targets.append(t_reg.cpu().numpy())
            
            pred_c_mat = torch.cat([torch.argmax(logits, dim=-1, keepdim=True) for logits in c], dim=1).cpu().numpy()
            all_cls_preds.append(pred_c_mat)
            all_cls_targets.append(t_cls.cpu().numpy())
            
    n_samples = len(val_loader.dataset)
    val_loss = val_loss_sum / n_samples
    val_reg = val_reg_sum / n_samples
    val_cls = val_cls_sum / n_samples
    
    v_preds_raw = np.concatenate(all_val_preds, axis=0)
    v_targets = np.concatenate(all_val_targets, axis=0)
    v_preds_post = soft_physics_postprocess(v_preds_raw)
    
    c_preds = np.concatenate(all_cls_preds, axis=0)
    c_targets = np.concatenate(all_cls_targets, axis=0)
    cls_acc = np.mean(c_preds == c_targets)
    
    r2_raw = calculate_competition_r2(v_targets, v_preds_raw)
    r2_post = calculate_competition_r2(v_targets, v_preds_post)
    return val_loss, val_reg, val_cls, r2_raw, r2_post, cls_acc, v_preds_post


In [ ]:
# 7. ConvNeXt-V2 Large 5-Fold Training Loop (with Gradient Accumulation)
train_df = load_and_pivot_data(CFG.TRAIN_CSV)
print(f'Train DataFrame ready: {len(train_df)} pasture plots.')

sgkf = StratifiedGroupKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
split_gen = sgkf.split(train_df, train_df[CFG.SPLIT_STRAT_COL], groups=train_df[CFG.SPLIT_GROUP_COL])

oof_preds = np.zeros((len(train_df), 5), dtype=np.float32)
fold_scores = []

for fold, (train_idx, val_idx) in enumerate(split_gen):
    print(f'\n======================================================')
    print(f'   TRAINING FOLD {fold + 1} / {CFG.N_FOLDS} [ConvNeXt-V2 Large]')
    print(f'======================================================')
    
    fold_train_df = train_df.iloc[train_idx].reset_index(drop=True)
    fold_val_df = train_df.iloc[val_idx].reset_index(drop=True)
    print(f'Train samples: {len(fold_train_df)} | Val samples: {len(fold_val_df)}')
    
    train_ds = DualStreamBiomassDataset(
        fold_train_df, CFG.TRAIN_IMG_DIR, CFG.IMG_SIZE, is_training=True,
        camera_scaling_prob=CFG.CAMERA_SCALE_PROB,
        strip_shuffle_prob=CFG.STRIP_SHUFFLE_PROB,
        view_swap_prob=CFG.VIEW_SWAP_PROB,
        grayscale_prob=CFG.GRAYSCALE_PROB
    )
    val_ds = DualStreamBiomassDataset(
        fold_val_df, CFG.TRAIN_IMG_DIR, CFG.IMG_SIZE, is_training=False,
        camera_scaling_prob=0.0, strip_shuffle_prob=0.0, view_swap_prob=0.0, grayscale_prob=0.0
    )
    
    train_loader = DataLoader(train_ds, batch_size=CFG.BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
    val_loader = DataLoader(val_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
    
    model = DualStreamBiomassModel(CFG.BACKBONE, pretrained=True).to(DEVICE)
    criterion = WeightedBiomassLoss()
    scaler = torch.amp.GradScaler('cuda')
    
    # ------------------------------------------------------------------
    # Stage 1: Freeze ConvNeXt Backbone -> Train Attention & MLP Heads
    # ------------------------------------------------------------------
    print(f'\n--- [Fold {fold + 1}] STAGE 1: Training Heads ({CFG.STAGE1_EPOCHS} epochs | LR: {CFG.LR}) ---')
    for p in model.backbone.parameters(): p.requires_grad = False
    optimizer = AdamW([p for p in model.parameters() if p.requires_grad], lr=CFG.LR, weight_decay=CFG.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG.STAGE1_EPOCHS, eta_min=CFG.LR * 0.1)
    
    best_r2 = -float('inf')
    best_val_preds = None
    checkpoint_name = f'best_model_{CFG.MODEL_TAG}_fold{fold+1}.pt'
    
    for epoch in range(1, CFG.STAGE1_EPOCHS + 1):
        t0 = time.time()
        model.train()
        tr_loss_sum, tr_reg_sum, tr_cls_sum = 0.0, 0.0, 0.0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc=f'Fold {fold+1} [S1 Ep {epoch:02d}/{CFG.STAGE1_EPOCHS:02d}]', leave=False)
        for step, batch in enumerate(pbar):
            img_l = batch['image_left'].to(DEVICE)
            img_r = batch['image_right'].to(DEVICE)
            t_reg = batch['targets'].to(DEVICE)
            t_cls = batch['targets_cls'].to(DEVICE)
            
            with torch.amp.autocast('cuda'):
                r, c = model(img_l, img_r)
                loss, l_reg, l_cls = criterion(r, c, t_reg, t_cls)
                loss = loss / CFG.GRAD_ACCUM_STEPS
                
            scaler.scale(loss).backward()
            
            if (step + 1) % CFG.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                if CFG.MAX_GRAD_NORM > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            tr_loss_sum += loss.item() * CFG.GRAD_ACCUM_STEPS * len(img_l)
            tr_reg_sum += l_reg.item() * len(img_l)
            tr_cls_sum += l_cls.item() * len(img_l)
            pbar.set_postfix({'loss': f'{loss.item()*CFG.GRAD_ACCUM_STEPS:.2f}', 'reg': f'{l_reg.item():.2f}'})
            
        scheduler.step()
        n_tr = len(train_loader.dataset)
        tr_loss, tr_reg, tr_cls = tr_loss_sum / n_tr, tr_reg_sum / n_tr, tr_cls_sum / n_tr
        val_loss, _, _, r2_raw, r2_post, cls_acc, v_post = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0
        
        is_best = r2_post > best_r2
        if is_best:
            best_r2 = r2_post
            best_val_preds = v_post
            torch.save(model.state_dict(), checkpoint_name)
            torch.save(model.state_dict(), f'best_model_fold{fold+1}.pt')
            
        star = '  ★ Best Saved' if is_best else ''
        print(f'[S1 Ep {epoch:02d}/{CFG.STAGE1_EPOCHS:02d}] Train: {tr_loss:.3f} | Val: {val_loss:.3f} | R2 Raw: {r2_raw:.4f} | R2 SoftBlend: {r2_post:.4f} | Cls Acc: {cls_acc:.1%} ({elapsed:.0f}s){star}')
        
    # ------------------------------------------------------------------
    # Stage 2: Unfreeze Backbone -> Full Fine-Tuning
    # ------------------------------------------------------------------
    print(f'\n--- [Fold {fold + 1}] STAGE 2: Full Fine-Tuning ({CFG.STAGE2_EPOCHS} epochs | Backbone LR: {CFG.LR*CFG.BACKBONE_LR_FACTOR:.1e}, Heads LR: {CFG.LR:.1e}) ---')
    for p in model.backbone.parameters(): p.requires_grad = True
    optimizer = AdamW([
        {'params': model.backbone.parameters(), 'lr': CFG.LR * CFG.BACKBONE_LR_FACTOR},
        {'params': [p for n, p in model.named_parameters() if not n.startswith('backbone')], 'lr': CFG.LR}
    ], weight_decay=CFG.WEIGHT_DECAY)
    scheduler = CosineAnnealingLR(optimizer, T_max=CFG.STAGE2_EPOCHS, eta_min=CFG.LR * 0.01)

    for epoch in range(1, CFG.STAGE2_EPOCHS + 1):
        t0 = time.time()
        model.train()
        tr_loss_sum, tr_reg_sum, tr_cls_sum = 0.0, 0.0, 0.0
        optimizer.zero_grad()
        
        pbar = tqdm(train_loader, desc=f'Fold {fold+1} [S2 Ep {epoch:02d}/{CFG.STAGE2_EPOCHS:02d}]', leave=False)
        for step, batch in enumerate(pbar):
            img_l = batch['image_left'].to(DEVICE)
            img_r = batch['image_right'].to(DEVICE)
            t_reg = batch['targets'].to(DEVICE)
            t_cls = batch['targets_cls'].to(DEVICE)
            
            with torch.amp.autocast('cuda'):
                r, c = model(img_l, img_r)
                loss, l_reg, l_cls = criterion(r, c, t_reg, t_cls)
                loss = loss / CFG.GRAD_ACCUM_STEPS
                
            scaler.scale(loss).backward()
            
            if (step + 1) % CFG.GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_loader):
                if CFG.MAX_GRAD_NORM > 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=CFG.MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad()
                
            tr_loss_sum += loss.item() * CFG.GRAD_ACCUM_STEPS * len(img_l)
            tr_reg_sum += l_reg.item() * len(img_l)
            tr_cls_sum += l_cls.item() * len(img_l)
            pbar.set_postfix({'loss': f'{loss.item()*CFG.GRAD_ACCUM_STEPS:.2f}', 'reg': f'{l_reg.item():.2f}'})
            
        scheduler.step()
        n_tr = len(train_loader.dataset)
        tr_loss, tr_reg, tr_cls = tr_loss_sum / n_tr, tr_reg_sum / n_tr, tr_cls_sum / n_tr
        val_loss, _, _, r2_raw, r2_post, cls_acc, v_post = evaluate(model, val_loader, criterion)
        elapsed = time.time() - t0
        
        is_best = r2_post > best_r2
        if is_best:
            best_r2 = r2_post
            best_val_preds = v_post
            torch.save(model.state_dict(), checkpoint_name)
            torch.save(model.state_dict(), f'best_model_fold{fold+1}.pt')
            
        star = '  ★ Best Saved' if is_best else ''
        print(f'[S2 Ep {epoch:02d}/{CFG.STAGE2_EPOCHS:02d}] Train: {tr_loss:.3f} | Val: {val_loss:.3f} | R2 Raw: {r2_raw:.4f} | R2 SoftBlend: {r2_post:.4f} | Cls Acc: {cls_acc:.1%} ({elapsed:.0f}s){star}')
            
    print(f'\n>>> Fold {fold+1} Finished! Best SoftBlend R2: {best_r2:.4f} <<<\n')
    oof_preds[val_idx] = best_val_preds
    fold_scores.append(best_r2)

overall_r2 = calculate_competition_r2(train_df[CFG.TARGET_ORDER].values, oof_preds)
print(f'\n======================================================')
print(f'>>> CONVNEXT-V2 5-FOLD OOF R2 SCORE: {overall_r2:.4f} <<<')
print(f'Per-Fold Scores: {[round(s, 4) for s in fold_scores]}')
print(f'======================================================')


In [ ]:
# 8. Test Set Inference with TTA & Standalone Submission Generation
test_csv_path = CFG.TEST_CSV
if not os.path.exists(test_csv_path) and os.path.exists('/kaggle/input'):
    for root, _, files in os.walk('/kaggle/input'):
        if 'test.csv' in files:
            test_csv_path = os.path.join(root, 'test.csv')
            break

print(f'Reading test data from: {test_csv_path}')
test_df_raw = pd.read_csv(test_csv_path)
if 'target_name' in test_df_raw.columns:
    test_df_raw['clean_id'] = test_df_raw['sample_id'].astype(str).apply(lambda x: x.split('__')[0])
    unique_test = test_df_raw[['clean_id', 'image_path']].drop_duplicates().reset_index(drop=True)
else:
    unique_test = test_df_raw.copy()
    if 'clean_id' not in unique_test.columns: unique_test['clean_id'] = unique_test['sample_id']

test_ds = DualStreamBiomassDataset(
    unique_test, CFG.TEST_IMG_DIR, CFG.IMG_SIZE, is_training=False,
    camera_scaling_prob=0.0, strip_shuffle_prob=0.0, view_swap_prob=0.0, grayscale_prob=0.0
)
test_loader = DataLoader(test_ds, batch_size=CFG.BATCH_SIZE, shuffle=False, num_workers=0)

model_checkpoints = sorted(glob.glob(f'best_model_{CFG.MODEL_TAG}_fold*.pt'))
if not model_checkpoints:
    model_checkpoints = sorted(glob.glob('best_model_fold*.pt'))
print(f'Found {len(model_checkpoints)} ConvNeXt-V2 checkpoints for ensemble: {model_checkpoints}')

all_fold_preds = []
for cp in model_checkpoints:
    model = DualStreamBiomassModel(CFG.BACKBONE, pretrained=False).to(DEVICE)
    model.load_state_dict(torch.load(cp, map_location=DEVICE))
    model.eval()
    
    f_preds = []
    with torch.no_grad():
        for batch in tqdm(test_loader, desc=f'Predicting {os.path.basename(cp)}', leave=False):
            img_l, img_r = batch['image_left'].to(DEVICE), batch['image_right'].to(DEVICE)
            # TTA: Standard + Mirrored Horizontal Flip
            r1, _ = model(img_l, img_r)
            r2, _ = model(torch.flip(img_r, [3]), torch.flip(img_l, [3]))
            avg_r = [(a + b) * 0.5 for a, b in zip(r1, r2)]
            f_preds.append(torch.cat(avg_r, dim=1).cpu().numpy())
    all_fold_preds.append(np.concatenate(f_preds, axis=0))

avg_raw = np.mean(all_fold_preds, axis=0)
avg_post = soft_physics_postprocess(avg_raw)

clean_ids = [s['sample_id'] for s in test_ds]
pred_dict = {
    clean_ids[i]: {col: avg_post[i, c_idx] for c_idx, col in enumerate(CFG.TARGET_ORDER)}
    for i in range(len(clean_ids))
}

submission_df = test_df_raw.copy()
submission_df['target'] = submission_df.apply(
    lambda r: pred_dict.get(r['clean_id'], {}).get(r['target_name'], 0.0), axis=1
)
sub = submission_df[['sample_id', 'target']]
sub.to_csv('submission.csv', index=False)
sub.to_csv('submission_convnextv2.csv', index=False)
print('ConvNeXt-V2 Submission saved to submission.csv. Preview:')
print(sub.head(10))
